In [2]:
#Install required packages
!pip install datasets requests pandas tqdm

In [17]:
#Load WildJailbreak dataset in streaming mode and preview first item
from datasets import load_dataset
from itertools import islice

dataset = load_dataset(
    "allenai/wildjailbreak",
    "train",
    split="train",
    streaming=True,
)

harmful_dataset = (x for x in dataset if "harmful" in x["data_type"])

first_five = list(islice(dataset, 5))
first_five[0]

In [15]:
#Set vLLM endpoint and model name constants
import requests
import pandas as pd
import uuid
from tqdm.auto import tqdm
from itertools import islice

VLLM_ENDPOINT = "http://192.168.2.229:8001/v1/chat/completions"
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

In [8]:
#Reload dataset and preview a single sample
dataset = load_dataset(
    "allenai/wildjailbreak",
    "train",
    split="train",
    streaming=True,
)

sample = next(iter(dataset))
sample

In [16]:
#Send first 25 dataset items to vLLM and collect responses into a DataFrame
import requests
import pandas as pd
import uuid
from tqdm.auto import tqdm
from itertools import islice

dataset = load_dataset(
    "allenai/wildjailbreak",
    "train",
    split="train",
    streaming=True,
)

results = []

for item in tqdm(islice(dataset, 25), total=25):

    prompt = item["adversarial"] if item["adversarial"] else item["vanilla"]
    session_id = str(uuid.uuid4())

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.2,
        "max_tokens": 256,
        "metadata": {"session_id": session_id}
    }

    try:
        r = requests.post(VLLM_ENDPOINT, json=payload, timeout=60)
        r.raise_for_status()
        content = r.json()["choices"][0]["message"]["content"]
    except Exception as e:
        content = f"ERROR: {e}"

    results.append({
        "session_id": session_id,
        "prompt": prompt,
        "response": content,
        "data_type": item["data_type"],
        "expected_refusal": item["completion"]
    })

df = pd.DataFrame(results)
df.head()

In [ ]:
#Save results to CSV and JSONL
df.to_csv("wildjailbreak_results_sample.csv", index=False)
df.to_json("wildjailbreak_results_sample.jsonl", orient="records", lines=True)

In [22]:
#Send all harmful items to vLLM, record responses and latency with abuse_label=1
import requests
import pandas as pd
import uuid
import time
from tqdm.auto import tqdm
from itertools import islice

results = []

for item in tqdm(harmful_dataset):

    prompt = item["adversarial"] if item["adversarial"] else item["vanilla"]
    session_id = str(uuid.uuid4())

    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.2,
        "max_tokens": 256
    }

    start = time.time()

    try:
        r = requests.post(VLLM_ENDPOINT, json=payload, timeout=60)
        r.raise_for_status()
        content = r.json()["choices"][0]["message"]["content"]
    except Exception as e:
        content = f"ERROR: {e}"

    latency = time.time() - start

    results.append({
        "session_id": session_id,
        "prompt": prompt,
        "response": content,
        "latency_seconds": latency,
        "data_type": item["data_type"],
        "expected_refusal": item["completion"],
        "abuse_label": 1
    })

    time.sleep(0.25)

In [23]:
#Reload dataset and filter to benign samples only
from datasets import load_dataset

dataset = load_dataset(
    "allenai/wildjailbreak",
    "train",
    split="train",
    streaming=True,
)

benign_dataset = (x for x in dataset if "benign" in x["data_type"])

In [26]:
#Send 8000 harmful items to vLLM, record responses with abuse_label=0
import requests
import pandas as pd
import uuid
import time
from tqdm.auto import tqdm

results = []

for item in tqdm(islice(harmful_dataset, 8000), total=8000):

    prompt = item["adversarial"] if item["adversarial"] else item["vanilla"]
    session_id = str(uuid.uuid4())

    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.2,
        "max_tokens": 256
    }

    start = time.time()

    try:
        r = requests.post(VLLM_ENDPOINT, json=payload, timeout=60)
        r.raise_for_status()
        content = r.json()["choices"][0]["message"]["content"]
    except Exception as e:
        content = f"ERROR: {e}"

    latency = time.time() - start

    results.append({
        "session_id": session_id,
        "prompt": prompt,
        "response": content,
        "latency_seconds": latency,
        "data_type": item["data_type"],
        "expected_refusal": item["completion"],
        "abuse_label": 0
    })

    time.sleep(0.25)